# 03 Finetuning ResNet18 (15-Class Baseline)

This notebook runs a baseline training loop using shared `src` modules.
It logs environment metadata, git commit hash, and split artifacts.


In [ ]:
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report
from torchvision.models import ResNet18_Weights, resnet18

from src.datasets import create_dataloaders
from src.experiment_log import ExperimentLog
from src.training import evaluate_model, save_model, train_model
from src.transforms import get_train_transform, get_val_transform
from src.utils import CLASS_NAMES, CSV_DIR, MODELS_DIR, NUM_CLASSES, PROJECT_ROOT, get_device


In [ ]:
BATCH_SIZE = 32
EPOCHS = 3
LR = 1e-4
SEED = 42

train_csv = str(Path(CSV_DIR) / "plantvillage_train.csv")
val_csv = str(Path(CSV_DIR) / "plantvillage_val.csv")
test_csv = str(Path(CSV_DIR) / "plantvillage_test.csv")

train_loader, val_loader, test_loader = create_dataloaders(
    train_csv=train_csv,
    val_csv=val_csv,
    test_csv=test_csv,
    train_transform=get_train_transform(),
    val_transform=get_val_transform(),
    label_column="class_label",
    batch_size=BATCH_SIZE,
    num_workers=0,
)

len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset)


In [ ]:
device = get_device()
print("Device:", device)

weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)


In [ ]:
model, history = train_model(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=EPOCHS,
    device=device,
    early_stop_patience=2,
)


In [ ]:
labels, preds, probs = evaluate_model(model, test_loader, device=device)
print(classification_report(labels, preds, target_names=CLASS_NAMES, digits=4))


In [ ]:
checkpoint_path = str(Path(MODELS_DIR) / "resnet18_15class_baseline.pth")
save_model(model, checkpoint_path)

log = ExperimentLog("resnet18_15class_baseline_seed42")
log.set_hyperparams(
    model="resnet18",
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    seed=SEED,
    label_column="class_label",
)
log.set_environment()
log.set_git_commit(repo_dir=PROJECT_ROOT)
log.set_split_artifacts(
    split_paths={"train": train_csv, "val": val_csv, "test": test_csv},
    seed=SEED,
    label_column="class_label",
    repo_dir=PROJECT_ROOT,
)
log.set_metrics(test_samples=len(labels))
log.save()
